# Sentimen Analisis — Modeling & Evaluasi

**Dataset:** Komentar YouTube tentang pelemahan Rupiah  
**Label:** positive / neutral / negative (LLM-labeled)  
**Tujuan:** Bangun model klasifikasi sentimen, evaluasi, simpan untuk deployment

## 1. Load & Bersihkan Data

In [ ]:
import pandas as pd, numpy as np, re, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib, warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../Data/Labeled(4)_comment.csv")
print(f"Total: {len(df)}")
print(df["sentiment"].value_counts())

In [ ]:
valid_labels = {"positive", "neutral", "negative"}
n_before = len(df)
df = df[df["sentiment"].str.strip().str.lower().isin(valid_labels)].copy()
df["sentiment"] = df["sentiment"].str.strip().str.lower()
print(f"Drop {n_before - len(df)} baris tidak valid -> {len(df)} tersisa")
print(df["sentiment"].value_counts())

## 2. Preprocessing Ringan

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"\\n|\\r", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"Setelah cleaning: {len(df)} komentar")

## 3. Train/Test Split + TF-IDF

In [ ]:
X = df["clean_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"\nDistribusi train:\n{y_train.value_counts()}")
print(f"\nDistribusi test:\n{y_test.value_counts()}")

In [ ]:
tfidf = TfidfVectorizer(
    max_features=2000, min_df=2, max_df=0.8,
    ngram_range=(1, 2), sublinear_tf=True
)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print(f"Shape train: {X_train_vec.shape}")
print(f"Shape test:  {X_test_vec.shape}")

## 4. Training 3 Model Baseline

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "SVM (Linear)": SVC(kernel="linear", probability=True, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    f1 = f1_score(y_test, y_pred, average="weighted")
    print(f"\n{'='*50}")
    print(f"{name} — F1 Weighted: {f1:.4f}")
    print(classification_report(y_test, y_pred, digits=4))
    print(confusion_matrix(y_test, y_pred))

## 5. Handle Imbalance dengan SMOTE

In [ ]:
smote_models = {
    "LR + SMOTE": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "NB + SMOTE": MultinomialNB(),
    "SVM + SMOTE": SVC(kernel="linear", probability=True, random_state=42, class_weight="balanced")
}

for name, clf in smote_models.items():
    pipeline = ImbPipeline([("smote", SMOTE(random_state=42, k_neighbors=1)), ("clf", clf)])
    pipeline.fit(X_train_vec, y_train)
    y_pred = pipeline.predict(X_test_vec)
    f1 = f1_score(y_test, y_pred, average="weighted")
    print(f"\n{'='*50}")
    print(f"{name} — F1 Weighted: {f1:.4f}")
    print(classification_report(y_test, y_pred, digits=4))
    print(confusion_matrix(y_test, y_pred))

## 6. Hyperparameter Tuning — Model Terbaik

In [ ]:
param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"]
}

pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42, k_neighbors=1)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])

grid = GridSearchCV(
    pipeline, param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="f1_weighted", n_jobs=-1, verbose=1
)
grid.fit(X_train_vec, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV F1: {grid.best_score_:.4f}")

y_pred = grid.predict(X_test_vec)
f1 = f1_score(y_test, y_pred, average="weighted")
print(f"\nTest F1 Weighted: {f1:.4f}")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred, labels=["negative", "neutral", "positive"])
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "neutral", "positive"],
            yticklabels=["negative", "neutral", "positive"])
plt.title("Confusion Matrix — Logistic Regression (Tuned)")
plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.tight_layout(); plt.show()

## 7. Error Analysis

In [ ]:
y_pred_best = grid.predict(X_test_vec)
error_df = pd.DataFrame({"text": X_test, "actual": y_test, "predicted": y_pred_best})
error_df = error_df[error_df["actual"] != error_df["predicted"]]

print(f"Total salah prediksi: {len(error_df)} dari {len(X_test)} ({len(error_df)/len(X_test)*100:.1f}%)")
print("\nSample 15 error:")
for i, row in error_df.sample(min(15, len(error_df)), random_state=42).iterrows():
    print(f"[A: {row['actual']:8s} | P: {row['predicted']:8s}] {row['text'][:80]}")

## 8. Simpan Model & Vectorizer

In [ ]:
import os
os.makedirs("../model", exist_ok=True)

joblib.dump(grid.best_estimator_, "../model/model.pkl")
joblib.dump(tfidf, "../model/tfidf.pkl")
joblib.dump({"labels": list(grid.best_estimator_.classes_)}, "../model/label_map.pkl")

print("Model tersimpan di ../model/")
print(f"Classes: {grid.best_estimator_.classes_}")

In [ ]:
# Verifikasi model bisa di-load dan prediksi
loaded_model = joblib.load("../model/model.pkl")
loaded_tfidf = joblib.load("../model/tfidf.pkl")

sample_texts = [
    "pemerintah ini bebal dan korup",
    "Terimakasih kontennya sangat edukatif",
    "menurut saya dollar sedang menguat terhadap rupiah"
]

for text in sample_texts:
    vec = loaded_tfidf.transform([clean_text(text)])
    pred = loaded_model.predict(vec)[0]
    proba = loaded_model.predict_proba(vec).max()
    print(f"[{pred}] ({proba:.2f}) {text}")

## Kesimpulan

- Logistic Regression + SMOTE + TF-IDF memberikan performa terbaik
- Kelas **positive** sangat minoritas eval perlu data lebih banyak
- Model dan vectorizer sudah disimpan untuk deployment Streamlit